# 🚀 Customer Churn Prediction REST API

## Objective

This notebook prepares our machine learning model for deployment.

The trained Random Forest model will be exposed using a REST API built with FastAPI.

The API will accept customer information as input and return churn predictions in real time.

---

## Why APIs?

A machine learning model inside Jupyter Notebook cannot be used by customers or businesses.

APIs allow external systems such as websites, dashboards, mobile applications, and CRM platforms to communicate with the model.

---

## Business Workflow

Customer Data

↓

REST API

↓

Machine Learning Model

↓

Prediction

↓

Business Action

---

## Expected Outcome

At the end of this notebook we will have a production-ready prediction API.

In [1]:
import pandas as pd
import numpy as np

import joblib

from fastapi import FastAPI
from pydantic import BaseModel

import uvicorn

In [2]:
model = joblib.load("customer_churn_model.pkl")

features = joblib.load("model_features.pkl")

C:\Users\Karan Singh\anaconda3\envs\customer_ai\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeClassifier from version 1.5.1 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
C:\Users\Karan Singh\anaconda3\envs\customer_ai\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator RandomForestClassifier from version 1.5.1 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [3]:
type(model)

sklearn.ensemble._forest.RandomForestClassifier

In [4]:
features

['price',
 'freight_value',
 'delivery_days',
 'total_spending',
 'purchase_frequency',
 'average_order_value',
 'average_review_score',
 'customer_lifetime_days',
 'recency_days']

In [5]:
app = FastAPI(
    title="Customer Churn Prediction API",
    description="Predict customer churn using Machine Learning",
    version="1.0"
)

In [24]:
class Customer(BaseModel):

    price: float

    freight_value: float

    delivery_days: float

    total_spending: float

    purchase_frequency: float

    average_review_score: float

    recency_days: float

    customer_lifetime_days: float

    average_order_value: float

In [25]:
@app.get("/")
def home():

    return {

        "Project": "Customer Churn Prediction API",

        "Author": "Your Name",

        "Status": "Running"

    }

In [26]:
app

In [27]:
@app.post("/predict")
def predict(customer: Customer):

    # Convert input to DataFrame
    input_data = pd.DataFrame([customer.dict()])

    # Keep feature order same as training
    input_data = input_data[features]

    # Predict
    prediction = model.predict(input_data)[0]

    # Probability
    probability = model.predict_proba(input_data)[0][1]

    return {

        "Prediction": int(prediction),

        "Churn Probability": round(float(probability),4)

    }

In [28]:
app.routes

[Route(path='/openapi.json', name='openapi', methods=['GET', 'HEAD']),
 Route(path='/docs', name='swagger_ui_html', methods=['GET', 'HEAD']),
 Route(path='/docs/oauth2-redirect', name='swagger_ui_redirect', methods=['GET', 'HEAD']),
 Route(path='/redoc', name='redoc_html', methods=['GET', 'HEAD']),
 APIRoute(path='/', name='home', methods=['GET']),
 APIRoute(path='/predict', name='predict', methods=['POST']),
 APIRoute(path='/', name='home', methods=['GET']),
 APIRoute(path='/predict', name='predict', methods=['POST'])]

In [29]:
sample = {

    "price":120,

    "freight_value":25,

    "delivery_days":6,

    "total_spending":800,

    "purchase_frequency":5,

    "average_review_score":4.5,

    "recency_days":20,

    "customer_lifetime_days":400,

    "average_order_value":160

}

In [47]:
sample_df = pd.DataFrame([sample])

sample_df = sample_df[features]

sample_df

,price,freight_value,delivery_days,total_spending,purchase_frequency,average_order_value,average_review_score,customer_lifetime_days,recency_days
0,120,25,6,420,4,105,4.8,400,30


In [48]:
prediction = model.predict(sample_df)[0]

probability = model.predict_proba(sample_df)[0][1]

print("Prediction:", prediction)

print("Probability:", probability)

Prediction: 0
Probability: 0.0


In [49]:
sample = {

    "price": 25,

    "freight_value": 80,

    "delivery_days": 30,

    "total_spending": 40,

    "purchase_frequency": 1,

    "average_review_score": 1.2,

    "recency_days": 400,

    "customer_lifetime_days": 20,

    "average_order_value": 40

}

In [50]:
sample_df = pd.DataFrame([sample])
sample_df = sample_df[features]

In [51]:
prediction = model.predict(sample_df)[0]

probability = model.predict_proba(sample_df)[0][1]

print(prediction)
print(probability)

1
0.55


In [53]:
if prediction == 1:
    print("⚠ Customer is likely to churn")
else:
    print("✅ Customer is likely to stay")

⚠ Customer is likely to churn


# Business Interpretation

The API returns two outputs:

- Prediction
    - 0 → Customer will stay
    - 1 → Customer will churn

- Churn Probability
    - Confidence score from the machine learning model.

Business teams can use this probability to prioritize retention campaigns.

Example:

Probability > 0.80

→ Send discount coupon

Probability between 0.50–0.80

→ Send promotional email

Probability < 0.50

→ No action required